In [1]:
#------------------------------------------------ Import Lib ----------------------------------------
import os
import datetime
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'LY CBLY' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd() ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

Running LY CBLY Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': []}

processdate = now.strftime('%Y-%m-%d')

# Lists from Jira DECD-6132
regdict = {
    1: {"ListName": "Commercial Banks",
        "URL": "https://cbl.gov.ly/en/banks/",
        "Comments": "Extract all entities"},
    2: {"ListName": "Representative Offices of Foreign Banks",
        "URL": "https://cbl.gov.ly/en/representative_offices_of_foreign_banks/",
        "Comments": "Extract all entities, there are 2 pages in this list"},
    3: {"ListName": "Electronic Payment",
        "URL": "https://cbl.gov.ly/en/electronic-payment/",
        "Comments": 'NEW LIST! Extract all entities under "Directory of approved electronic payment companies"'},
}

ListLabeldict = {1: 1, 2: 1, 3: 4}  # 1 = bank list; 3 (e-payment) = everything else

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'}

In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def get_soup(url):
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return BeautifulSoup(r.text, 'html.parser')

def cell_text(td):
    """The data-filter attribute holds the clean value (and bypasses the Cloudflare
    email obfuscation shown in the visible text); fall back to the visible text
    without the mobile header label span ("Bank:", "Address:", ...).
    A lone dash is the site's empty-value placeholder."""
    val = td.get('data-filter')
    if val is None:
        for span in td.select('span.fs-thead-stacked'):
            span.decompose()
        val = td.get_text(' ', strip=True)
    val = val.strip()
    return '' if val in {'-', '–', '—'} else val

def add_row(rowdata):
    for key in sqldict:
        sqldict[key].append(rowdata.get(key, ''))

def norm_date(text, fmts):
    text = text.strip()
    for fmt in fmts:
        try:
            return datetime.datetime.strptime(text, fmt).strftime('%Y-%m-%d')
        except ValueError:
            pass
    return text  # keep verbatim if the site shows something unparseable

# home country of the foreign mother bank (list 2, "Country" column)
country_iso = {'Bahrain': 'BH', 'The United Arab Emirates': 'AE', 'Jordan': 'JO', 'England': 'GB',
               'Italy': 'IT', 'Malta': 'MT', 'Egypt': 'EG', 'Austria': 'AT', 'Tunisia': 'TN',
               'Morocco': 'MA', 'Germany': 'DE'}

def common_fields(listnr):
    return {'ListLabel': ListLabeldict[listnr],
            'RegCtry': 'LY',
            'RegCode': 'CBLY',
            'ListCode': str(listnr),
            'ListName': regdict[listnr]['ListName'],
            'ListLanguage': 'EN',
            'ListProcessDate': processdate,
            'RegulationType': 'Regulated',
            'Cntry': 'LY'}

In [5]:
#------------------------------------------------ Begin_Main : List 1 - Commercial Banks ----------------------------------------
listnr = 1
print(f"[INFO] : Working 1/3 _({regdict[listnr]['ListName']})_ ")

soup = get_soup(regdict[listnr]['URL'])
rows = soup.find('table').find_all('tr')[1:]  # row 0 = header; the "load more" button only reveals rows already in the HTML
for tr in rows:
    tds = tr.find_all('td')
    name, address, phone, fax, email, website = (cell_text(td) for td in tds[:6])
    link = tds[5].find('a')
    if link and link.get('href'):
        website = link['href']
    row = common_fields(listnr)
    row.update({'Name': name, 'Address_1': address, 'Phone': phone, 'Fax': fax,
                'Email': email, 'Website': website})
    add_row(row)

print(f"[INFO] : List 1 -> {len(rows)} entities")

[INFO] : Working 1/3 _(Commercial Banks)_ 


[INFO] : List 1 -> 25 entities


In [6]:
#------------------------------------------------ Begin_Main : List 2 - Representative Offices of Foreign Banks ----------------------------------------
listnr = 2
print(f"[INFO] : Working 2/3 _({regdict[listnr]['ListName']})_ ")

page = 1
count = 0
while True:
    url = regdict[listnr]['URL'] if page == 1 else regdict[listnr]['URL'] + f'?sf_paged={page}'
    soup = get_soup(url)
    rows = soup.find('table').find_all('tr')[1:]
    if not rows:
        break
    for tr in rows:
        tds = tr.find_all('td')
        name = cell_text(tds[0])
        country = cell_text(tds[1])
        approval_date = cell_text(tds[2])   # "Approval Date / CBL"; the "Office Opening Date" column (tds[3]) has no sqldict field
        phone = cell_text(tds[4])
        row = common_fields(listnr)
        row.update({'Name': name, 'Phone': phone,
                    'RegulationDate': norm_date(approval_date, ['%B %d, %Y']),
                    'Cntry - Mother company': country_iso.get(country, country)})
        add_row(row)
    count += len(rows)
    if not soup.select_one(f'ul.uk-pagination a[href*="sf_paged={page + 1}"]'):
        break
    page += 1

print(f"[INFO] : List 2 -> {count} entities across {page} page(s)")

[INFO] : Working 2/3 _(Representative Offices of Foreign Banks)_ 


[INFO] : List 2 -> 18 entities across 2 page(s)


In [7]:
#------------------------------------------------ Begin_Main : List 3 - Electronic Payment ----------------------------------------
listnr = 3
print(f"[INFO] : Working 3/3 _({regdict[listnr]['ListName']})_ ")

soup = get_soup(regdict[listnr]['URL'])
# the only table on the page sits under the "Directory of approved electronic payment companies" heading
rows = soup.find('table').find_all('tr')[1:]
for tr in rows:
    tds = tr.find_all('td')
    name, status, activity, decision_no, decision_date = (cell_text(td) for td in tds[:5])
    if status != 'Licensed':
        print(f"[WARN] : '{name}' has License Status '{status}' (not 'Licensed') - review RegulationType")
    row = common_fields(listnr)
    row.update({'Name': name, 'License_Type': activity,
                'InternalID_1': decision_no,
                'InternalID_1_type': 'Decision No.' if decision_no else '',
                'RegulationDate': norm_date(decision_date, ['%d/%m/%Y'])})
    add_row(row)

print(f"[INFO] : List 3 -> {len(rows)} entities")

[INFO] : Working 3/3 _(Electronic Payment)_ 


[INFO] : List 3 -> 13 entities


In [8]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print('Saved {} rows to {}'.format(len(df), os.path.join(scriptfolder, filename)))

Saved 56 rows to /Users/wuj1/Library/CloudStorage/OneDrive-Moody's/Desktop/Regulator/LY CBLY/LY CBLY SQL Ready 2026-07-13 09.55.36.xlsx
